# 03 Docker与部署 · 第02课：Dockerfile与镜像构建

> 本课把第01课里的现成镜像使用，推进到自己编写构建说明：Dockerfile。你会构建一个真实的健康检查服务镜像，并用容器访问它。

## 1. 连接上一课与本课目标

上一课的`nginx:alpine`是别人已经构建好的镜像。本课回答：镜像里的Python程序和文件从哪里来？

学完后应能：

1. 解释构建上下文、基础镜像和镜像层；
2. 逐行看懂`FROM`、`WORKDIR`、`COPY`、`RUN`、`EXPOSE`和`CMD`；
3. 为标准库健康检查服务写Dockerfile；
4. 使用`.dockerignore`排除本地环境和缓存；
5. 构建镜像、运行容器并访问`/health`；
6. 根据构建输出定位文件、命令和上下文问题。

## 2. 环境准备

本课需要Docker Engine和pytest。课程目录中已经提供`status_app.py`、Dockerfile骨架、`.dockerignore`和测试。

```bash
cd "03_Docker与部署/02_Dockerfile与镜像构建"
python -m pip install -r requirements.txt
docker version
```

如果`docker version`同时显示Client和Server，说明Docker Engine可用。

## 3. 构建上下文与基础镜像

执行：

```bash
docker build -t lesson-status:v1 .
```

`build`是构建子命令；`-t lesson-status:v1`给新镜像命名并打标签；最后的`.`是构建上下文，表示把当前课程目录作为构建输入。Dockerfile中的`COPY`只能读取构建上下文内且未被`.dockerignore`排除的文件。

`FROM python:3.14-slim`选择基础镜像。它不是在本机激活虚拟环境，而是从一个包含Python运行时的镜像开始创建新的镜像层。标签明确写出版本，避免每次构建得到不可解释的基础环境变化。

## 4. Dockerfile指令逐行解释

```dockerfile
FROM python:3.14-slim
WORKDIR /app
COPY status_app.py .
EXPOSE 8000
CMD ["python", "status_app.py"]
```

- `FROM`：指定基础镜像，每个普通Dockerfile通常从它开始；
- `WORKDIR`：设置后续指令和默认进程的工作目录，不需要手写`cd`；
- `COPY`：把构建上下文中的文件复制到镜像内；`.`表示当前工作目录，也就是`/app`；
- `EXPOSE`：记录镜像预期使用的容器端口，不等于把端口发布到宿主机；
- `CMD`：容器启动时的默认命令。JSON数组是exec形式，第一项是程序，后面是参数。

本课没有`RUN`，因为健康服务只使用Python标准库，不需要安装第三方包。后续有依赖时，才会在Dockerfile中加入复制依赖清单和`RUN python -m pip install ...`。

## 5. 镜像层和构建缓存

Docker会按Dockerfile顺序处理指令。前面没有变化的步骤通常可以复用缓存，后面步骤再继续执行。因此有依赖的项目通常先复制依赖清单并安装，再复制频繁变化的源代码。

本课只有一个源文件，先观察完整构建，再改动`status_app.py`后重新构建，留意输出中的`CACHED`。缓存加速构建，但不能把缓存误当成程序已经重新执行；需要新结果时可以使用`docker build --no-cache`进行对照。

## 6. `.dockerignore`不是`.gitignore`

`.gitignore`控制哪些文件进入Git；`.dockerignore`控制哪些文件不发送给Docker构建上下文。两者目的相似但作用对象不同。

本课至少排除：`.git`、`.venv`、`__pycache__`和`*.pyc`。虚拟环境可能很大，Git目录会泄露无关历史，缓存没有构建价值。构建上下文越小，上传和构建越快。

## 7. 从镜像运行健康服务

构建成功后运行：

```bash
docker run -d --name lesson-status -p 8080:8000 lesson-status:v1
docker ps
curl http://127.0.0.1:8080/health
docker logs lesson-status
```

`-p 8080:8000`左边是宿主机端口，右边是容器端口。服务代码必须监听`0.0.0.0`，否则只监听容器内部的回环地址时，宿主机转发请求无法到达。

清理本课资源：

```bash
docker stop lesson-status
docker rm lesson-status
docker image rm lesson-status:v1
```

## 8. 课堂练习：完成Dockerfile

打开目录中的Dockerfile和`.dockerignore`，完成所有TODO。

验收目标：

- 基础镜像为`python:3.14-slim`；
- 工作目录为`/app`；
- 只复制`status_app.py`；
- 声明8000端口；
- 使用`CMD ["python", "status_app.py"]`；
- `.dockerignore`包含`.git`、`.venv`、`__pycache__`和`*.pyc`。

先运行静态测试，再构建真实镜像。测试不会替你写入任何文件。

## 9. 构建故障排查

- `failed to read dockerfile`：命令最后的构建上下文不对，或Dockerfile不在上下文根目录；
- `COPY failed`：文件不在上下文中，文件名大小写不一致，或被`.dockerignore`排除了；
- 容器立即退出：检查`docker ps -a`和`docker logs lesson-status`，通常是`CMD`写错或主程序启动失败；
- 容器运行但curl失败：检查应用监听地址、容器端口和`-p`两侧顺序；
- 构建结果没有反映代码修改：检查缓存提示，必要时使用`docker build --no-cache`对照。

## 10. 课末作业与测试

本课作业就是完成目录中的Dockerfile和`.dockerignore`，不需要修改`status_app.py`或测试文件。

先运行：

```bash
python -m pytest test_Dockerfile.py -v
```

再运行真实构建：

```bash
docker build -t lesson-status:v1 .
docker run -d --name lesson-status -p 8080:8000 lesson-status:v1
curl http://127.0.0.1:8080/health
```

预期HTTP响应为`{"status": "ok"}`。测试覆盖Dockerfile关键指令、exec形式CMD、`.dockerignore`条目和标准库服务本身的健康接口。

## 11. 理解题

1. 构建命令最后的`.`为什么重要？
2. `FROM`和`COPY`分别从哪里取得内容？
3. `EXPOSE 8000`和`docker run -p 8080:8000`有什么不同？
4. 为什么`CMD`推荐使用`["python", "status_app.py"]`这种exec形式？
5. `.dockerignore`和`.gitignore`分别影响什么？
6. 为什么修改源代码后重新构建，前面的步骤仍可能显示`CACHED`？
7. 容器启动后立即退出时，应先看哪些信息？

### 理解题答案

1. `.`指定当前目录为构建上下文，Docker只能读取上下文中且未被忽略的文件；上下文错了，`COPY`就找不到材料。
2. `FROM`从本地或镜像仓库取得基础镜像；`COPY`从构建上下文把文件复制进新镜像层。
3. `EXPOSE`只是镜像元数据中的端口说明；`-p`才在创建容器时建立宿主机端口到容器端口的发布映射。
4. exec形式把程序和参数作为独立列表传给容器主进程，信号和退出状态更容易正确传递，也不会额外经过shell解析。
5. `.dockerignore`影响发送给Docker的构建上下文；`.gitignore`影响Git是否跟踪文件。
6. Docker按指令顺序复用没有变化的前置层；源代码只影响后面的`COPY`层，所以前面的基础镜像和准备步骤可能仍可缓存。
7. 先用`docker ps -a`确认状态，再用`docker logs`读取主进程输出，随后检查CMD、工作目录、文件复制和端口配置。

## 12. 小结

本课把一个Python服务从源文件变成可运行镜像：选择基础镜像，设置工作目录，复制代码，声明端口，指定主进程，再用端口发布启动容器。下一课将把这一套方法迁移到FastAPI应用，并处理依赖安装与更合理的缓存顺序。